In [1]:
from pathlib import Path
import pandas as pd

In [15]:
# BASE = Path("/scratch/shelvia.wongso/cbo_results/adamw/cifar10_resnet20") # adamw
# BASE = Path("/scratch/shelvia.wongso/cbo_results/sgd/cifar10_resnet20") # sgd
# BASE = Path("/scratch/shelvia.wongso/cbo_results/ivon_price/cifar10_resnet20") # ivon
# BASE = Path("/scratch/shelvia.wongso/cbo_results/ucbopt/cifar10_resnet20") # ucbopt
BASE = Path("/scratch/shelvia.wongso/cbo_results/ucbopt/cifar10_resnet20") # mcdrop

METRICS = ["acc", "acc@5", "nll", "ece","brier"]

rows = []

LRS = ["5e-2", "5e-4", "1e-3", "5e-3"] # adamw
LRS = ["1e-4", "1e-1", "2e-1", "3e-1", "4e-1"] # sgd
LRS = ["0.5", "1.0", "1.5", "2.0", "2.5", "3.0", "3.5"] # ivon, ucbopt

nr = 0.2

rows = []

for lr in LRS:
    # pattern = f"lr_{lr}_wd_2e-4_tf_0.2_nr_{nr}" # adamw, sgd
    # pattern = f"lr_{lr}_wd_2e-4_h0_0.5_ess_50000_tf_0.2_nr_{nr}" # ivon
    pattern = f"lr_{lr}_wd_2e-4_h0_0.5_curv_0.0_tf_0.2_nr_{nr}" # ucbopt
    matched_dirs = sorted(BASE.glob(pattern))

    if not matched_dirs:
        print(f"[WARN] No folder found for lr={lr}")
        continue

    for lr_dir in matched_dirs:
        seed_best_rows = []

        for seed in range(5):
            seed_dir = lr_dir / f"seed={seed}"
            if not seed_dir.exists():
                print(f"[WARN] Missing {seed_dir}")
                continue

            run_dirs = sorted([d for d in seed_dir.iterdir() if d.is_dir()])
            if not run_dirs:
                print(f"[WARN] No run dirs in {seed_dir}")
                continue

            run_dir = run_dirs[-1]   # latest run
            test_csv = run_dir / "test.csv"
            if not test_csv.exists():
                print(f"[WARN] Missing {test_csv}")
                continue

            df = pd.read_csv(test_csv)

            best_idx = df["nll"].idxmin()
            best_row = df.loc[best_idx].copy()
            best_row["lr"] = lr
            best_row["lr_dir"] = lr_dir.name
            best_row["seed"] = seed
            seed_best_rows.append(best_row)

        if not seed_best_rows:
            continue

        seed_best_df = pd.DataFrame(seed_best_rows)
        summary = {
            "lr": lr,
            "lr_dir": lr_dir.name,
            "num_seeds": len(seed_best_df),
        }

        for m in METRICS:
            if m in seed_best_df.columns:
                summary[f"{m}_mean"] = seed_best_df[m].mean()
                summary[f"{m}_std"] = seed_best_df[m].std(ddof=1)

        rows.append(summary)

summary_df = pd.DataFrame(rows).sort_values("nll_mean").reset_index(drop=True)

cols = [
    "lr", "num_seeds",
    "acc_mean", "acc_std",
    "acc@5_mean", "acc@5_std",
    "nll_mean", "nll_std",
    "ece_mean", "ece_std",
    "brier_mean", "brier_std",
]
cols = [c for c in cols if c in summary_df.columns]

print(summary_df[cols].to_string(index=False))

 lr  num_seeds  acc_mean  acc_std  acc@5_mean  acc@5_std  nll_mean  nll_std  ece_mean  ece_std  brier_mean  brier_std
3.0          5   0.71348 0.003124     0.96560   0.005297  0.918313 0.016882  0.087064 0.030938    0.407463   0.005391
3.5          5   0.71504 0.002633     0.96160   0.005959  0.922161 0.007033  0.087953 0.029935    0.406689   0.005525
2.5          5   0.70710 0.004037     0.96014   0.004464  0.937006 0.011463  0.085864 0.023146    0.412916   0.004098
1.5          5   0.70384 0.013814     0.95746   0.004656  0.950806 0.025393  0.081828 0.031222    0.418191   0.011799
2.0          5   0.70170 0.010638     0.96072   0.003713  0.955116 0.013020  0.090577 0.020954    0.421115   0.009262
1.0          5   0.69822 0.013709     0.95616   0.006158  0.969660 0.038034  0.085323 0.017317    0.426190   0.015957
0.5          5   0.68220 0.017623     0.95824   0.006478  0.984886 0.037685  0.055248 0.024949    0.439196   0.016162


In [16]:
display_df = summary_df[cols].copy()

for col in display_df.columns:
    if col in ["lr", "lr_dir", "num_seeds"]:
        continue
    elif "acc" in col:
        display_df[col] = display_df[col].apply(lambda x: f"{x*100:.2f}" if pd.notnull(x) else "")
    else:
        display_df[col] = display_df[col].apply(lambda x: f"{x:.3f}" if pd.notnull(x) else "")

print(display_df.to_string(index=False))

 lr  num_seeds acc_mean acc_std acc@5_mean acc@5_std nll_mean nll_std ece_mean ece_std brier_mean brier_std
3.0          5    71.35    0.31      96.56      0.53    0.918   0.017    0.087   0.031      0.407     0.005
3.5          5    71.50    0.26      96.16      0.60    0.922   0.007    0.088   0.030      0.407     0.006
2.5          5    70.71    0.40      96.01      0.45    0.937   0.011    0.086   0.023      0.413     0.004
1.5          5    70.38    1.38      95.75      0.47    0.951   0.025    0.082   0.031      0.418     0.012
2.0          5    70.17    1.06      96.07      0.37    0.955   0.013    0.091   0.021      0.421     0.009
1.0          5    69.82    1.37      95.62      0.62    0.970   0.038    0.085   0.017      0.426     0.016
0.5          5    68.22    1.76      95.82      0.65    0.985   0.038    0.055   0.025      0.439     0.016
